In [67]:
# parse the position data into a pandas DataFrame
# past option position list here

position_txt = '''
MRNA 11/06/2026 205.00 P

-1	$10.025	-$1.3564	-11.92%	-$1,002.50	+$21.82	+2.13%	
-$1,024.32
+$21.82	+2.13%	N/A	
MRNA 11/13/2026 185.00 P

-1	$6.35	-$0.7712	-10.83%	-$635.00	-$15.67	-2.53%	
-$619.33
-$15.67	-2.53%	N/A	
MRNA 11/20/2026 185.00 P

-1	$6.925	-$0.80	-10.36%	-$692.50	+$14.83	+2.1%	
-$707.33
+$14.83	+2.1%	N/A	
MRNA 11/27/2026 180.00 P

-1	$7.575	-$0.0843	-1.1%	-$757.50	-$58.17	-8.32%	
-$699.33
-$58.17	-8.32%	N/A	

'''

import re
import pandas as pd
import datetime


def to_float(s):
    """Convert strings like '$2.15', '-$1.30', '+$130.36' to float."""
    s = s.strip()
    sign = -1 if s.startswith('-') else 1
    s = s.lstrip('+-').lstrip('$')
    return sign * float(s)

positions = []

blocks = re.split(r'\n(?=[A-Z]+ \d{2}/\d{2}/\d{4})', position_txt.strip())

for block in blocks:
    lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
    if not lines:
        continue

    header = lines[0]
    m = re.match(r'(\w+)\s+(\d{2}/\d{2}/\d{4})\s+([\d.]+)\s+([PC])', header)
    if not m:
        continue
    ticker, expiry, strike, opt_type = m.groups()

    data_text = ' '.join(lines[1:])
    tokens = data_text.replace(',', '').split()

    if len(tokens) < 10:
        continue

    record = {
        'Ticker':       ticker,
        'Expiry':       datetime.datetime.strptime(expiry, "%m/%d/%Y").date(),
        'Strike':       float(strike),
        'Type':         opt_type,
        #'Qty':          int(tokens[0]),
        'price':        to_float(tokens[1]),
        #'Price Chng $': to_float(tokens[2]),
        #'Price Chng %': tokens[3].rstrip('%'),
        'Mkt Val':      to_float(tokens[4]),
        'Day Chng $':   to_float(tokens[5]),
        'Day Chng %':   tokens[6].rstrip('%'),
        'Cost Basis':   to_float(tokens[7]),
        'Gain/Loss $':  to_float(tokens[8]),
        'Gain/Loss %':  tokens[9].rstrip('%'),
        #'Reinvest?':    tokens[10] if len(tokens) > 10 else 'N/A',
    }
    positions.append(record)

pd.DataFrame(positions)


,Ticker,Expiry,Strike,Type,price,Mkt Val,Day Chng $,Day Chng %,Cost Basis,Gain/Loss $,Gain/Loss %
0,MRNA,2026-11-06,205.0,P,10.025,-1002.5,21.82,+2.13,-1024.32,21.82,+2.13
1,MRNA,2026-11-13,185.0,P,6.350,-635.0,-15.67,-2.53,-619.33,-15.67,-2.53
2,MRNA,2026-11-20,185.0,P,6.925,-692.5,14.83,+2.1,-707.33,14.83,+2.1
3,MRNA,2026-11-27,180.0,P,7.575,-757.5,-58.17,-8.32,-699.33,-58.17,-8.32


In [68]:
# enrich with open date

positions[0]['Open Date'] = datetime.datetime.strptime('10/21/2026', "%m/%d/%Y").date()
positions[1]['Open Date'] = datetime.datetime.strptime('10/21/2026', "%m/%d/%Y").date()
positions[2]['Open Date'] = datetime.datetime.strptime('10/21/2026', "%m/%d/%Y").date()
positions[3]['Open Date'] = datetime.datetime.strptime('10/21/2026', "%m/%d/%Y").date()
pd.DataFrame(positions)

,Ticker,Expiry,Strike,Type,price,Mkt Val,Day Chng $,Day Chng %,Cost Basis,Gain/Loss $,Gain/Loss %,Open Date
0,MRNA,2026-11-06,205.0,P,10.025,-1002.5,21.82,+2.13,-1024.32,21.82,+2.13,2026-10-21
1,MRNA,2026-11-13,185.0,P,6.350,-635.0,-15.67,-2.53,-619.33,-15.67,-2.53,2026-10-21
2,MRNA,2026-11-20,185.0,P,6.925,-692.5,14.83,+2.1,-707.33,14.83,+2.1,2026-10-21
3,MRNA,2026-11-27,180.0,P,7.575,-757.5,-58.17,-8.32,-699.33,-58.17,-8.32,2026-10-21


In [100]:
strike_txt = '''
Oct. 23, 2026 (W) (Fri: 2 days)
170.00		0.01	0.09	0.03	-0.16	24	578	
Select

172.50		0.01	0.10	0.04	-0.17	28	181	
Select

175.00		0.03	0.09	0.04	-0.18	59	1,225	
Select

177.50		0.01	0.12	0.04	-0.19	7	148	
Select

180.00		0.01	0.10	0.05	-0.19	60	1,706	
Select

182.50		0.04	0.11	0.07	-0.21	36	329	
Select

185.00		0.05	0.15	0.10	-0.23	35	709	
Select

187.50		0.07	0.16	0.11	-0.26	59	435	
Select


Oct. 30, 2026 (W) (Fri: 9 days)
170.00		2.00	2.45	2.33	-0.14	58	1,353	
Select

172.50		2.00	2.45	2.43	-0.31	15	309	
Select

175.00		2.17	3.10	2.60	-0.41	44	1,212	
Select

177.50		2.52	3.40	2.88	-0.48	23	60	
Select

180.00		3.00	3.45	3.20	-0.51	47	2,342	
Select

182.50		2.91	3.90	3.65	-0.44	56	144	
Select

185.00		3.40	4.05	3.70	-0.78	26	199	
Select

187.50		3.80	5.05	4.20	-0.69	8	229	
Select


Nov. 06, 2026 (W) (Fri: 16 days)
170.00		2.63	3.70	3.08	-0.60	6	62	
Select

172.50		3.00	4.20	6.17	+2.09	0	17	
Select

175.00		3.35	4.20	4.25	-0.32	133	71	
Select

177.50		3.15	5.55	7.47	+2.41	0	10	
Select

180.00		4.05	5.15	4.60	-0.95	7	156	
Select

182.50		4.30	7.40	5.32	-0.72	1	11	
Select

185.00		4.80	5.75	5.25	-1.28	5	62	
Select

187.50		5.20	6.85	6.61	-0.40	1	30	
Select


Nov. 13, 2026 (W) (Fri: 23 days)
170.00		2.70	4.20	3.50	-0.56	2	91	
Select

172.50		2.82	6.45	9.30	+4.79	0	4	
Select

175.00		4.20	5.15	4.73	-0.23	1	32	
Select

177.50		3.65	5.95	8.51	+3.01	0	18	
Select

180.00		4.20	5.95	5.24	-0.80	3	66	
Select

182.50		4.35	8.25	10.17	+3.59	0	14	
Select

185.00		5.80	6.90	6.35	-0.77	16	48	
Select

187.50		5.40	8.65	13.00	+5.34	0	4	
Select


Nov. 20, 2026 (Fri: 30 days)
160.00		2.79	3.15	2.96	-0.29	73	2,787	
Select

165.00		3.40	3.75	3.55	-0.35	11	1,784	
Select

170.00		4.05	4.30	4.35	-0.33	538	6,130	
Select

175.00		4.80	5.50	5.00	-0.57	16	592	
Select

180.00		5.70	6.45	6.10	-0.47	190	2,053	
Select

185.00		6.55	7.40	7.01	-0.71	14	2,597	
Select

190.00		7.80	8.50	8.21	-0.79	47	2,581	
Select

192.50		8.20	9.45	10.05	+0.37	0	9	
Select


Nov. 27, 2026 (W) (Fri: 37 days)
160.00		2.25	5.25	3.51	-0.78	2	34	
Select

165.00		2.55	6.30	4.19	-0.78	1	31	
Select

170.00		4.40	6.10	5.38	-0.26	1	57	
Select

175.00		4.45	6.10	9.12	+2.47	0	20	
Select

180.00		6.15	6.60	6.70	-0.96	5	53	
Select

185.00		7.00	8.95	8.31	-0.42	3	33	
Select

190.00		8.10	9.80	9.35	-0.53	8	157	
Select

195.00		8.75	11.65	12.80	+1.48	3	24	
Select

'''

In [101]:

import re
import pandas as pd

DATE_PAT = re.compile(r'^[A-Z][a-z]+\.\s+\d+,\s+\d{4}')

strikes = []
current_date = None

for line in strike_txt.splitlines():
    line = line.strip()
    if not line or line == "Select":
        continue
    if DATE_PAT.match(line):
        # 取行首的日期部分，忽略括号里的注释
        date_str = DATE_PAT.match(line).group(0)
        current_date = pd.to_datetime(date_str).date()
        continue
    cols = [c.strip() for c in line.split('\t')]
    if len(cols) < 4 or current_date is None:
        continue
    try:
        strike_price = float(cols[0].replace(',', ''))
        col2         = float(cols[2].replace(',', ''))
        strikes.append({'Date': current_date, 'Strike': strike_price, 'Bid': col2})
    except ValueError:
        continue

pd.DataFrame(strikes)[:5]


,Date,Strike,Bid
0,2026-10-23,170.0,0.01
1,2026-10-23,172.5,0.01
2,2026-10-23,175.0,0.03
3,2026-10-23,177.5,0.01
4,2026-10-23,180.0,0.01


In [102]:
%load_ext autoreload
%autoreload 2

from plot_options import plot_timeline
plot_timeline(positions, strikes)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<IPython.core.display.Javascript object>

In [ ]:
from plot_options import plot_strikes_3d
plot_strikes_3d(strikes)